# L8 - Code development: pytest and pip

## Q2: My own test

- Pick a piece of python code that you like (your own PhD project, or take one of the exercises from this class). Implement a unit test.
- Put it under git version control, and write a github action that runs the tests at every commit.
- Edit the github options to make sure the code *cannot* be committed if the tests fail (that's a common thing for big projects with many collaborations, nobody is allowed to break the code)



In [1]:
f = open("test_poker.py", "r")
print(f.read())
f.close()

import poker

def test_deck_cards():
    deck = poker.Deck()
    assert len(deck.cards) == 52

def test_get_cards():
    deck = poker.Deck()
    hand = deck.get_cards(5)
    assert len(hand) == 5
    assert len(deck.cards) == 47

def test_three_of_a_kind():
    # remember that in class Card: def __init__(self, suit=1, rank=2)
    # I want to take 5 cards
    hand = [poker.Card(1, 4), poker.Card(2, 4), poker.Card(3, 10), poker.Card(3, 4), poker.Card(2, 11)]
    assert poker.three_of_a_kind(hand) == True



In [2]:
import os
os.system('pytest test_poker.py -v')

============================= test session starts ==============================
platform linux -- Python 3.11.9, pytest-7.4.4, pluggy-1.0.0 -- /home/bmoreschi/anaconda3/bin/python
cachedir: .pytest_cache
rootdir: /home/bmoreschi/Desktop/projects/repo/scientificcomputing_bicocca_2025/working
plugins: anyio-4.2.0
collecting ... collected 3 items

test_poker.py::test_deck_cards PASSED                                    [ 33%]
test_poker.py::test_get_cards PASSED                                     [ 66%]
test_poker.py::test_three_of_a_kind PASSED                               [100%]

============================== 3 passed in 0.02s ===============================


0

In order to push to main I need to create another branch and open a pull request. I setup github to allow merging on main only through pull request and only if the test passes. 
If one of the tests doesn't pass, the pull request doesn't allow me to merge on main.

## Q4: So slow

Pick a piece of your code that you dislike. It's sooooo slooooow. 
- Profile it.
- Find the bottleneck.
- Rewrite that bottleneck making a better use of numpy arrays, or jitting it with Numba.
- Enjoy your faster code

In [3]:
import sys, pickle
import numpy as np

sys.path.insert(0, "pta_research") # path to the code that I need for PTA analysis
import discovery as ds

with open("pta_research/psrs.pkl", "rb") as f: # I take 100 pulsars that I really used in my PTA analysis
    psrs = pickle.load(f)

pos = np.array([p.pos for p in psrs])

In PTA analysis the overlap reduction function (ORF) is used to compute the correlation between the timing residuals of pairs of pulsars. 

The ORF depends on the angular separation between the two pulsars.

For an isotropic gravitational wave background (GWB) the ORF is the so called Hellings and Downs (HD).

In [ ]:
def orf_hd(pos1, pos2): # position of 2 pulsars of the array
    d = (1.0 - pos1 @ pos2)/2.0
    return 0.5 - (d/2.0)*(0.5 - 3.0*np.log(d)) # equation for computing the ORF for one pulsar pair

# we have to compute the ORF for all pulsars pairs
def orf_matrix_slow(pos):
    npsr = len(pos)
    gamma = np.ones((npsr, npsr))
    for a in range(npsr):
        for b in range(a+1, npsr):
            v = orf_hd(pos[a], pos[b])
            gamma[a, b] = gamma[b, a] = v
    return gamma

When we run the PTMCMC sampler to search for the GWB signal modeled with HD, we have to compute orf_matrix_slow for each iteration (1e6 times), thus the code is very slow. Unfortunately I cannot run the slow version in my laptop because it requires hours.

So, just to show the improvement in speed, I present here another case where still we have to compute the orf_matrix_slow many times, but only 1000 times and not a million.

Usually in PTA to test if the GWB signal is present in the data we shuffle the pulsars in the sky (sky scramble) and compute the ORF matrix again.

In [5]:
# here we create random positions in the sky for the pulsars in the array
def random_sky(N):
    # spherical coordinates
    theta = np.arccos(np.random.uniform(-1, 1, N))
    phi = np.random.uniform(-np.pi, np.pi, N)

    sky = np.zeros((N, 3))
    # cartesian coordinates
    sky[:, 0] = np.sin(theta)*np.cos(phi)
    sky[:, 1] = np.sin(theta)*np.sin(phi)
    sky[:, 2] = np.cos(theta)
    return sky

def sky_scrambles(orf_matrix, scrambles):
    for k in range(scrambles):
        orf_matrix(random_sky(len(pos)))

In [6]:
import cProfile

# Usually in PTA analysis we call the sky_scramble 1000 times
cProfile.run('sky_scrambles(orf_matrix_slow, 1000)', sort='cumtime')


         4960004 function calls in 27.846 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000   27.846   27.846 {built-in method builtins.exec}
        1    0.000    0.000   27.846   27.846 <string>:1(<module>)
        1    0.008    0.008   27.846   27.846 816274744.py:14(sky_scrambles)
     1000   10.496    0.010   27.780    0.028 2361859903.py:10(orf_matrix_slow)
  4950000   17.259    0.000   17.259    0.000 2361859903.py:5(orf_hd)
     1000    0.033    0.000    0.056    0.000 816274744.py:2(random_sky)
     1000    0.019    0.000    0.023    0.000 numeric.py:136(ones)
     2000    0.020    0.000    0.020    0.000 {method 'uniform' of 'numpy.random.mtrand.RandomState' objects}
     2000    0.004    0.000    0.004    0.000 {built-in method builtins.len}
     1000    0.003    0.000    0.003    0.000 {built-in method numpy.zeros}
     1000    0.002    0.000    0.002    0.000 {built-in method numpy.e

The bottleneck is the orf_hd function

In [7]:
def orf_matrix_fast(pos):
    d = (1.0 - pos @ pos.T)/2.0 # here we speed up because we compute d for all pulsars pairs at once

    # the diagonal of d computed in the previous step is 0, but in the next lines we do the log of d, so we have to change the diagonal to another number
    for a in range(len(pos)):
        d[a, a] = 1.0

    gamma = 0.5 - (d/2.0)*(0.5 - 3.0*np.log(d)) # we compute the ORF for all pulsars pairs at once
    for a in range(len(pos)):
        gamma[a, a] = 1.0 # I have to set the diagonal to 1 because the autocorrelation of the same pulsar is 1
    return gamma

In [8]:
import time

t0 = time.time()
sky_scrambles(orf_matrix_slow, 1000)
t1 = time.time()
print("slow: {:.3f} s".format(t1-t0))

t0 = time.time()
sky_scrambles(orf_matrix_fast, 1000)
t1 = time.time()
print("fast: {:.3f} s".format(t1-t0))

slow: 10.219 s
fast: 0.097 s
